## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [44]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_anthropic import ChatAnthropic   
from langchain_ollama import ChatOllama
# from langchain_google_gemini import ChatGoogleGenerativeAI
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [45]:
# MODEL = "gpt-4o"
MODEL =  "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [46]:
# If you want to use OLLAMA, you can do so by changing the following line:
# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# If you want to use openai, you can do so by changing the following line:  

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!


**What is temperature?**

Parameter that controls how much variety shoud be there in the LLM response. A higher temperature means more variety.

temperture = 0 means pick the token with the highest probability


In [47]:
# retriever is a function that takes a query and returns a list of documents
retriever = vectorstore.as_retriever()

#### These LangChain objects implement the method `invoke()`

In [48]:
retriever.invoke("Who is Avery?")

[Document(id='67a33807-5bdb-4c5f-bca3-68c37c467b13', metadata={'doc_type': 'employees', 'source': 'knowledge-base/employees/Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and ada

In [49]:
# llm is a function that takes a list of messages and returns a response
# MODEL = "gpt-3.5-turbo"
llm = ChatOpenAI(temperature=0, model_name=MODEL)
# llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)
response = llm.invoke("Who is Avery?")
print (response.content)

Could you please provide more context or specify which Avery you are referring to? There are many individuals and characters named Avery.


## Time to put this together!

Note: Better than the temperature, a well constructed System prompt ensures accuracy

In [50]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [51]:
# THis is the RAG piece of logic 

def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [52]:
answer_question("Who is Avery Lancaster?", [])

'Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm. She has been with the company since its founding in 2015 and has played a key role in establishing Insurellm as a leading provider in the insurance technology industry. Avery is known for her innovative leadership, risk management expertise, and her commitment to diversity, inclusion, and community engagement. She is based in San Francisco, California.'

## What could possibly come next? 😂

In [53]:
gr.ChatInterface(answer_question).launch()

/Users/kashyaprajpurohit/myGitHubRepos/kr-aws-data-ml-labs/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!